# Prerequisite: Setup Target Catalog for Apps Migration

This notebook sets up the target catalog with necessary schemas, volumes, and copies tables from the source catalog.

Run this BEFORE deploying apps to the target workspace/catalog.

## Configuration

In [ ]:
try:
    source_catalog = dbutils.widgets.get("source_catalog")
except:
    source_catalog = "YOUR_SOURCE_CATALOG"

try:
    target_catalog = dbutils.widgets.get("target_catalog")
except:
    target_catalog = "YOUR_TARGET_CATALOG"

try:
    schema = dbutils.widgets.get("schema")
except:
    schema = "apps_demo"

try:
    volume = dbutils.widgets.get("volume")
except:
    volume = "apps_migration"

print(f"Source catalog: {source_catalog}")
print(f"Target catalog: {target_catalog}")
print(f"Schema: {schema}")
print(f"Volume: {volume}")

## Create target schema and volume

In [ ]:
print(f"Creating schema {target_catalog}.{schema}...")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{schema}")
print("Schema created.")

print(f"\nCreating volume {target_catalog}.{schema}.{volume}...")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {target_catalog}.{schema}.{volume}")
print("Volume created.")

## Discover tables in source schema

In [ ]:
print(f"Listing tables in {source_catalog}.{schema}...")

try:
    tables_df = spark.sql(f"SHOW TABLES IN {source_catalog}.{schema}")
    tables = [row.tableName for row in tables_df.collect()]
    print(f"Found {len(tables)} table(s): {tables}")
except Exception as e:
    tables = []
    print(f"No tables found or schema doesn't exist: {e}")

## Copy tables to target catalog

In [ ]:
copied_tables = []
skipped_tables = []
failed_tables = []

for table in tables:
    source_table = f"{source_catalog}.{schema}.{table}"
    target_table = f"{target_catalog}.{schema}.{table}"
    
    print(f"\nProcessing: {table}")
    
    # Check if target table already exists
    try:
        exists = spark.sql(f"SHOW TABLES IN {target_catalog}.{schema} LIKE '{table}'").count() > 0
    except:
        exists = False
    
    if exists:
        print(f"  Skipping - already exists in target")
        skipped_tables.append(table)
        continue
    
    try:
        # Copy table using CREATE TABLE AS SELECT
        print(f"  Copying {source_table} -> {target_table}")
        spark.sql(f"CREATE TABLE {target_table} AS SELECT * FROM {source_table}")
        copied_tables.append(table)
        print(f"  Copied successfully")
    except Exception as e:
        print(f"  FAILED: {e}")
        failed_tables.append({"table": table, "error": str(e)})

print("\n" + "=" * 60)

## Copy views (if any)

In [ ]:
print(f"Listing views in {source_catalog}.{schema}...")

try:
    views_df = spark.sql(f"""
        SELECT table_name 
        FROM {source_catalog}.information_schema.views 
        WHERE table_schema = '{schema}'
    """)
    views = [row.table_name for row in views_df.collect()]
    print(f"Found {len(views)} view(s): {views}")
except Exception as e:
    views = []
    print(f"No views found: {e}")

copied_views = []
for view in views:
    source_view = f"{source_catalog}.{schema}.{view}"
    target_view = f"{target_catalog}.{schema}.{view}"
    
    print(f"\nProcessing view: {view}")
    
    try:
        # Get view definition
        view_def_df = spark.sql(f"SHOW CREATE TABLE {source_view}")
        view_def = view_def_df.collect()[0][0]
        
        # Replace source catalog with target catalog in the view definition
        new_view_def = view_def.replace(source_catalog, target_catalog)
        
        # Create view in target
        spark.sql(new_view_def)
        copied_views.append(view)
        print(f"  Created view successfully")
    except Exception as e:
        print(f"  FAILED: {e}")

## Summary

In [ ]:
print("=" * 60)
print("TARGET CATALOG SETUP SUMMARY")
print("=" * 60)
print(f"Target catalog: {target_catalog}")
print(f"Target schema: {target_catalog}.{schema}")
print(f"Target volume: {target_catalog}.{schema}.{volume}")
print()
print(f"Tables copied: {len(copied_tables)}")
if copied_tables:
    for t in copied_tables:
        print(f"  - {t}")
print(f"Tables skipped (already exist): {len(skipped_tables)}")
if skipped_tables:
    for t in skipped_tables:
        print(f"  - {t}")
print(f"Tables failed: {len(failed_tables)}")
if failed_tables:
    for t in failed_tables:
        print(f"  - {t['table']}: {t['error']}")
print(f"Views copied: {len(copied_views)}")
if copied_views:
    for v in copied_views:
        print(f"  - {v}")
print()
print("Target catalog is ready for app migration.")
print("=" * 60)

## Verify target schema contents

In [ ]:
print(f"\nContents of {target_catalog}.{schema}:")
print("\nTables:")
display(spark.sql(f"SHOW TABLES IN {target_catalog}.{schema}"))

print("\nVolumes:")
display(spark.sql(f"SHOW VOLUMES IN {target_catalog}.{schema}"))